# 快速上手：受体 + 配体库 → 候选清单

本 Notebook 演示本作品的最小可复现流程，与 `README.md` §3 的命令行入口等价：

1. 检查运行环境与依赖版本；
2. 用示例数据运行一键入口 `screen.py`（真实对接，约数秒）；
3. 读取 `results/results.csv`，查看候选清单并按亲和力排序；
4. 复核运行日志（参数、随机种子、引擎版本、耗时）。

> 说明：默认使用内置 AutoDock Vina，CPU 即可运行；不需要网络，也不需要配置大模型。

In [ ]:
import platform, sys, importlib.metadata as meta

print("Python", platform.python_version(), "|", platform.platform())
for pkg in ("vina", "rdkit", "meeko", "numpy"):
    try:
        print(f"{pkg}=={meta.version(pkg)}")
    except Exception as exc:  # 缺依赖时如实提示
        print(f"{pkg}: 未安装（{exc}）")

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "../screen.py",
    "--receptor", "../data/example/receptor_demo.pdb",
    "--ligands",  "../data/example/ligands_demo.smi",
    "--center", "31.5,13.74,24.36", "--size", "22,22,22",
    "--exhaustiveness", "1", "--n-poses", "1", "--seed", "42",
    "--out", "../results/results.csv", "--log-dir", "../logs",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True, cwd=".")

In [ ]:
import csv

# 只用标准库读取结果清单：提交环境无需额外依赖
with open("../results/results.csv", encoding="utf-8") as fh:
    rows = list(csv.DictReader(fh))

for row in rows[:10]:
    print(f"{row['candidate_id']:>6}  {row['name'][:28]:<28} "
          f"{row['affinity_kcal_mol']:>7} kcal/mol  LE={row['ligand_efficiency']:<7} "
          f"{row['engine']} ({row['engine_version']})  {row['status']}")


In [ ]:
import json

log = json.load(open("../logs/run_example.json", encoding="utf-8"))
log

## 用自己的数据运行

把 `--receptor` / `--ligands` 换成自己的文件即可；不给位点盒时加 `--auto-site`，
需要 GPU 引擎时在配置中登记可执行文件后用 `--engine external`。

结果清单字段与《候选清单》模板的对应关系见 `README.md` §4。